In [ ]:

import os
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from dataset import ClockDataset

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Using device: {device}")
Using device: mps
CNN
# ===================== 1. Define the model (CNN) =====================
class DigitalClockNet(nn.Module):
    def __init__(self):
        super(DigitalClockNet, self).__init__()
        
        # First convolutional block: basic feature extraction (edges, corners)
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Make it 64x64
        )
        
        # Second convolutional block: more complex features
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Reduce to 32x32
        )
        
        # Third convolutional block: high-level features
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Reduce to 16x16
        )

        # Fully connected layers for regression
        # Input size: 128 channels * 16 * 16 spatial dimensions
        self.fc = nn.Sequential( 
            nn.Flatten(), # Flatten the tensor
            nn.Linear(128 * 16 * 16, 512), # Fully connected layer 
            nn.ReLU(),
            nn.Dropout(0.5), # Reduce overfitting
            nn.Linear(512, 3) # Output: (h, m, s)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x # tensor of shape (batch_size, 3)
# ===================== 2. Training function =====================
def train_model(data_dir, num_epochs=10, batch_size=32, learning_rate=0.0001):
    # Setting up device
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    # Data transformations and loaders
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
    ])
    
    train_dataset = ClockDataset(data_dir, subset='train', transform=transform)
    test_dataset = ClockDataset(data_dir, subset='test', transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Making the model 
    model = DigitalClockNet().to(device)
    
    # Loss Function: MSE (Mean Squared Error) for regression
    criterion = nn.MSELoss()
    
    # Optimizer: Adam optimizer
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_loss = float('inf')
    os.makedirs('checkpoints', exist_ok=True)

    print("Starting training...")
    
    for epoch in range(num_epochs):
        model.train() # Set model to training mode
        running_loss = 0.0
        
        for batch in train_loader:
            images = batch['digital_img'].to(device)
            labels = batch['time_label'].to(device) # Normalized (h, m, s)
            
            # 1. Forward pass
            outputs = model(images)
            
            # 2. Calculate loss
            loss = criterion(outputs, labels)
            
            # 3. Backward pass and optimization
            optimizer.zero_grad() # Clear gradients
            loss.backward() # Backpropagation
            optimizer.step() # Update weights
            
            running_loss += loss.item() # Update running loss

        avg_train_loss = running_loss / len(train_loader) 

        # Calculate test loss
        model.eval()
        test_loss = 0.0
        with torch.no_grad(): # No gradient calculation during evaluation
            for batch in test_loader:
                images = batch['digital_img'].to(device)
                labels = batch['time_label'].to(device)
                outputs = model(images)
                test_loss += criterion(outputs, labels).item()
        
        avg_test_loss = test_loss / len(test_loader)
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")

        # Save the best model
        if avg_test_loss < best_loss:
            best_loss = avg_test_loss
            torch.save(model.state_dict(), "checkpoints/digital_reader_best.pth")
            print("  -> Saved best model!")
# ===================== 3. Run =====================
if __name__ == "__main__":
    # Start training with specified data directory and epochs 
    train_model(data_dir="../data", num_epochs=100)
Using device: mps
Starting training...
Epoch [1/100], Train Loss: 0.1036, Test Loss: 0.0889
  -> Saved best model!
Epoch [2/100], Train Loss: 0.0919, Test Loss: 0.0854
  -> Saved best model!
Epoch [3/100], Train Loss: 0.0925, Test Loss: 0.0860
Epoch [4/100], Train Loss: 0.0911, Test Loss: 0.0870
Epoch [5/100], Train Loss: 0.0910, Test Loss: 0.0879
Epoch [6/100], Train Loss: 0.0913, Test Loss: 0.0886
Epoch [7/100], Train Loss: 0.0901, Test Loss: 0.0883
Epoch [8/100], Train Loss: 0.0896, Test Loss: 0.0864
Epoch [9/100], Train Loss: 0.0900, Test Loss: 0.0859
Epoch [10/100], Train Loss: 0.0902, Test Loss: 0.0857
Epoch [11/100], Train Loss: 0.0888, Test Loss: 0.0862
Epoch [12/100], Train Loss: 0.0888, Test Loss: 0.0856
Epoch [13/100], Train Loss: 0.0894, Test Loss: 0.0870
Epoch [14/100], Train Loss: 0.0888, Test Loss: 0.0850
  -> Saved best model!
Epoch [15/100], Train Loss: 0.0875, Test Loss: 0.0856
Epoch [16/100], Train Loss: 0.0874, Test Loss: 0.0842
  -> Saved best model!
Epoch [17/100], Train Loss: 0.0869, Test Loss: 0.0831
  -> Saved best model!
Epoch [18/100], Train Loss: 0.0858, Test Loss: 0.0820
  -> Saved best model!
Epoch [19/100], Train Loss: 0.0848, Test Loss: 0.0805
  -> Saved best model!
Epoch [20/100], Train Loss: 0.0828, Test Loss: 0.0790
  -> Saved best model!
Epoch [21/100], Train Loss: 0.0803, Test Loss: 0.0769
  -> Saved best model!
Epoch [22/100], Train Loss: 0.0778, Test Loss: 0.0745
  -> Saved best model!
Epoch [23/100], Train Loss: 0.0756, Test Loss: 0.0711
  -> Saved best model!
Epoch [24/100], Train Loss: 0.0734, Test Loss: 0.0685
  -> Saved best model!
Epoch [25/100], Train Loss: 0.0711, Test Loss: 0.0650
  -> Saved best model!
Epoch [26/100], Train Loss: 0.0673, Test Loss: 0.0635
  -> Saved best model!
Epoch [27/100], Train Loss: 0.0656, Test Loss: 0.0603
  -> Saved best model!
Epoch [28/100], Train Loss: 0.0631, Test Loss: 0.0579
  -> Saved best model!
Epoch [29/100], Train Loss: 0.0616, Test Loss: 0.0567
  -> Saved best model!
Epoch [30/100], Train Loss: 0.0581, Test Loss: 0.0530
  -> Saved best model!
Epoch [31/100], Train Loss: 0.0558, Test Loss: 0.0528
  -> Saved best model!
Epoch [32/100], Train Loss: 0.0551, Test Loss: 0.0501
  -> Saved best model!
Epoch [33/100], Train Loss: 0.0528, Test Loss: 0.0488
  -> Saved best model!
Epoch [34/100], Train Loss: 0.0514, Test Loss: 0.0466
  -> Saved best model!
Epoch [35/100], Train Loss: 0.0503, Test Loss: 0.0455
  -> Saved best model!
Epoch [36/100], Train Loss: 0.0491, Test Loss: 0.0443
  -> Saved best model!
Epoch [37/100], Train Loss: 0.0476, Test Loss: 0.0435
  -> Saved best model!
Epoch [38/100], Train Loss: 0.0462, Test Loss: 0.0413
  -> Saved best model!
Epoch [39/100], Train Loss: 0.0453, Test Loss: 0.0423
Epoch [40/100], Train Loss: 0.0438, Test Loss: 0.0412
  -> Saved best model!
Epoch [41/100], Train Loss: 0.0427, Test Loss: 0.0381
  -> Saved best model!
Epoch [42/100], Train Loss: 0.0411, Test Loss: 0.0375
  -> Saved best model!
Epoch [43/100], Train Loss: 0.0396, Test Loss: 0.0350
  -> Saved best model!
Epoch [44/100], Train Loss: 0.0383, Test Loss: 0.0341
  -> Saved best model!
Epoch [45/100], Train Loss: 0.0373, Test Loss: 0.0334
  -> Saved best model!
Epoch [46/100], Train Loss: 0.0352, Test Loss: 0.0318
  -> Saved best model!
Epoch [47/100], Train Loss: 0.0337, Test Loss: 0.0307
  -> Saved best model!
Epoch [48/100], Train Loss: 0.0330, Test Loss: 0.0285
  -> Saved best model!
Epoch [49/100], Train Loss: 0.0310, Test Loss: 0.0277
  -> Saved best model!
Epoch [50/100], Train Loss: 0.0299, Test Loss: 0.0265
  -> Saved best model!
Epoch [51/100], Train Loss: 0.0293, Test Loss: 0.0253
  -> Saved best model!
Epoch [52/100], Train Loss: 0.0285, Test Loss: 0.0245
  -> Saved best model!
Epoch [53/100], Train Loss: 0.0279, Test Loss: 0.0234
  -> Saved best model!
Epoch [54/100], Train Loss: 0.0266, Test Loss: 0.0224
  -> Saved best model!
Epoch [55/100], Train Loss: 0.0254, Test Loss: 0.0223
  -> Saved best model!
Epoch [56/100], Train Loss: 0.0252, Test Loss: 0.0221
  -> Saved best model!
Epoch [57/100], Train Loss: 0.0248, Test Loss: 0.0202
  -> Saved best model!
Epoch [58/100], Train Loss: 0.0243, Test Loss: 0.0190
  -> Saved best model!
Epoch [59/100], Train Loss: 0.0233, Test Loss: 0.0184
  -> Saved best model!
Epoch [60/100], Train Loss: 0.0229, Test Loss: 0.0184
  -> Saved best model!
Epoch [61/100], Train Loss: 0.0219, Test Loss: 0.0168
  -> Saved best model!
Epoch [62/100], Train Loss: 0.0210, Test Loss: 0.0170
Epoch [63/100], Train Loss: 0.0202, Test Loss: 0.0154
  -> Saved best model!
Epoch [64/100], Train Loss: 0.0197, Test Loss: 0.0155
Epoch [65/100], Train Loss: 0.0183, Test Loss: 0.0139
  -> Saved best model!
Epoch [66/100], Train Loss: 0.0180, Test Loss: 0.0129
  -> Saved best model!
Epoch [67/100], Train Loss: 0.0167, Test Loss: 0.0132
Epoch [68/100], Train Loss: 0.0165, Test Loss: 0.0134
Epoch [69/100], Train Loss: 0.0157, Test Loss: 0.0110
  -> Saved best model!
Epoch [70/100], Train Loss: 0.0148, Test Loss: 0.0099
  -> Saved best model!
Epoch [71/100], Train Loss: 0.0149, Test Loss: 0.0101
Epoch [72/100], Train Loss: 0.0141, Test Loss: 0.0088
  -> Saved best model!
Epoch [73/100], Train Loss: 0.0133, Test Loss: 0.0081
  -> Saved best model!
Epoch [74/100], Train Loss: 0.0136, Test Loss: 0.0082
Epoch [75/100], Train Loss: 0.0121, Test Loss: 0.0071
  -> Saved best model!
Epoch [76/100], Train Loss: 0.0120, Test Loss: 0.0072
Epoch [77/100], Train Loss: 0.0112, Test Loss: 0.0065
  -> Saved best model!
Epoch [78/100], Train Loss: 0.0108, Test Loss: 0.0063
  -> Saved best model!
Epoch [79/100], Train Loss: 0.0105, Test Loss: 0.0061
  -> Saved best model!
Epoch [80/100], Train Loss: 0.0106, Test Loss: 0.0059
  -> Saved best model!
Epoch [81/100], Train Loss: 0.0105, Test Loss: 0.0055
  -> Saved best model!
Epoch [82/100], Train Loss: 0.0099, Test Loss: 0.0054
  -> Saved best model!
Epoch [83/100], Train Loss: 0.0100, Test Loss: 0.0047
  -> Saved best model!
Epoch [84/100], Train Loss: 0.0098, Test Loss: 0.0054
Epoch [85/100], Train Loss: 0.0094, Test Loss: 0.0043
  -> Saved best model!
Epoch [86/100], Train Loss: 0.0090, Test Loss: 0.0040
  -> Saved best model!
Epoch [87/100], Train Loss: 0.0092, Test Loss: 0.0039
  -> Saved best model!
Epoch [88/100], Train Loss: 0.0088, Test Loss: 0.0042
Epoch [89/100], Train Loss: 0.0086, Test Loss: 0.0039
  -> Saved best model!
Epoch [90/100], Train Loss: 0.0087, Test Loss: 0.0039
  -> Saved best model!
Epoch [91/100], Train Loss: 0.0083, Test Loss: 0.0036
  -> Saved best model!
Epoch [92/100], Train Loss: 0.0081, Test Loss: 0.0034
  -> Saved best model!
Epoch [93/100], Train Loss: 0.0082, Test Loss: 0.0030
  -> Saved best model!
Epoch [94/100], Train Loss: 0.0081, Test Loss: 0.0034
Epoch [95/100], Train Loss: 0.0079, Test Loss: 0.0030
  -> Saved best model!
Epoch [96/100], Train Loss: 0.0082, Test Loss: 0.0033
Epoch [97/100], Train Loss: 0.0079, Test Loss: 0.0030
Epoch [98/100], Train Loss: 0.0081, Test Loss: 0.0027
  -> Saved best model!
Epoch [99/100], Train Loss: 0.0074, Test Loss: 0.0027
  -> Saved best model!
Epoch [100/100], Train Loss: 0.0072, Test Loss: 0.0025
  -> Saved best model!
def denormalize_time(tensor_time):
    h = int(tensor_time[0].item() * 23)
    m = int(tensor_time[1].item() * 59)
    s = int(tensor_time[2].item() * 59)
    return f"{h:02d}:{m:02d}:{s:02d}"

def evaluate():
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Loading the test dataset for evaluation
    test_dataset = ClockDataset("../data", subset='test', transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

    model = DigitalClockNet().to(device) # Load the trained model
    try:
        model.load_state_dict(torch.load("checkpoints/digital_reader_best.pth", map_location=device))
        print("CNN Model loaded successfully!")
    except FileNotFoundError:
        print("Error: Checkpoint not found.")
        return

    model.eval() # Set model to evaluation mode

    print("\n--- Visual Evaluation CNN ---")
    print(f"{'Actual':<10} | {'Predicted':<10} | {'Diff'}")
    print("-" * 35)

    with torch.no_grad(): # No gradient calculation during evaluation
        for i, batch in enumerate(test_loader):
            if i >= 10: break
            
            img = batch['digital_img'].to(device)
            target = batch['time_label'].to(device)
            
            output = model(img)
            
            actual_str = denormalize_time(target[0])
            pred_str = denormalize_time(output[0])
            
            # Calculate loss for reporting
            loss = torch.nn.functional.mse_loss(output, target).item()
            
            print(f"{actual_str:<10} | {pred_str:<10} | {loss:.4f}")

if __name__ == "__main__":
    evaluate()
CNN Model loaded successfully!

--- Visual Evaluation CNN ---
Actual     | Predicted  | Diff
-----------------------------------
01:25:17   | -4:35:28   | 0.0402
02:11:52   | 38:186:134 | 4.4077
21:21:21   | 68:253:85  | 6.9487
18:06:21   | 68:188:108 | 5.4186
22:05:40   | 79:181:137 | 5.9500
09:40:15   | 12:26:04   | 0.0325
11:48:13   | 12:48:04   | 0.0086
18:02:33   | 67:184:112 | 5.2530
08:02:59   | 59:224:110 | 6.5739
11:22:21   | 56:231:104 | 6.1133
Res-Net
# ===================== 1. Define the ResNet Model =====================
class DigitalClockResNet(nn.Module):
    def __init__(self):
        super(DigitalClockResNet, self).__init__()
        
        # Load a pre-trained ResNet18 model
        # "DEFAULT" weights mean it was trained on ImageNet (millions of images)
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # We need to replace the last layer (fc) because ResNet outputs 1000 classes by default.
        # We want it to output 3 numbers (Hour, Minute, Second).
        
        num_features = self.model.fc.in_features # usually 512 for ResNet18
        
        self.model.fc = nn.Sequential(
            nn.Linear(num_features, 256), # Compress features
            nn.ReLU(),
            nn.Dropout(0.3),              # Prevent overfitting [cite: 46]
            nn.Linear(256, 3)             # Output: (h, m, s)
        )

    def forward(self, x):
        return self.model(x)
# ===================== 2. ResNet Training Function =====================
def train_resnet_model(data_dir, num_epochs=20, batch_size=32, learning_rate=0.0001):
    # Setting up device
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    # TRANSFORMATIONS: ResNet requires 224x224 and specific normalization
    transform = transforms.Compose([
        transforms.Resize((224, 224)), 
        transforms.ToTensor(),
        # Standard ImageNet normalization (Mean, Std)
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Load Data
    # Note: Ensure data_dir points to the folder containing 'train' and 'test' folders
    train_dataset = ClockDataset(data_dir, subset='train', transform=transform)
    test_dataset = ClockDataset(data_dir, subset='test', transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize ResNet
    model = DigitalClockResNet().to(device)
    
    # Loss and Optimizer
    criterion = nn.MSELoss()
    # Lower learning rate (0.0001) is better for fine-tuning
    optimizer = optim.Adam(model.parameters(), lr=learning_rate) 

    best_loss = float('inf')
    os.makedirs('checkpoints', exist_ok=True)

    print("Starting ResNet training...")
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for batch in train_loader:
            images = batch['digital_img'].to(device)
            labels = batch['time_label'].to(device)
            
            # Forward -> Loss -> Backward
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader) 

        # Validation
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                images = batch['digital_img'].to(device)
                labels = batch['time_label'].to(device)
                outputs = model(images)
                test_loss += criterion(outputs, labels).item()
        
        avg_test_loss = test_loss / len(test_loader)
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")

        # Save Best Model
        if avg_test_loss < best_loss:
            best_loss = avg_test_loss
            # Save with a distinct name so we don't overwrite the CNN
            torch.save(model.state_dict(), "checkpoints/digital_resnet_best.pth")
            print("  -> Saved best ResNet model!")

# Run the training
# Make sure data_dir is correct (e.g., "./data" or "../data")
if __name__ == "__main__":
    train_resnet_model(data_dir="../data", num_epochs=100)
Using device: mps
Starting ResNet training...
Epoch [1/100], Train Loss: 0.1023, Test Loss: 0.0709
  -> Saved best ResNet model!
Epoch [2/100], Train Loss: 0.0505, Test Loss: 0.0400
  -> Saved best ResNet model!
Epoch [3/100], Train Loss: 0.0277, Test Loss: 0.0172
  -> Saved best ResNet model!
Epoch [4/100], Train Loss: 0.0189, Test Loss: 0.0105
  -> Saved best ResNet model!
Epoch [5/100], Train Loss: 0.0170, Test Loss: 0.0126
Epoch [6/100], Train Loss: 0.0140, Test Loss: 0.0172
Epoch [7/100], Train Loss: 0.0150, Test Loss: 0.0103
  -> Saved best ResNet model!
Epoch [8/100], Train Loss: 0.0137, Test Loss: 0.0104
Epoch [9/100], Train Loss: 0.0133, Test Loss: 0.0097
  -> Saved best ResNet model!
Epoch [10/100], Train Loss: 0.0128, Test Loss: 0.0057
  -> Saved best ResNet model!
Epoch [11/100], Train Loss: 0.0124, Test Loss: 0.0087
Epoch [12/100], Train Loss: 0.0128, Test Loss: 0.0055
  -> Saved best ResNet model!
Epoch [13/100], Train Loss: 0.0109, Test Loss: 0.0081
Epoch [14/100], Train Loss: 0.0114, Test Loss: 0.0050
  -> Saved best ResNet model!
Epoch [15/100], Train Loss: 0.0107, Test Loss: 0.0058
Epoch [16/100], Train Loss: 0.0101, Test Loss: 0.0032
  -> Saved best ResNet model!
Epoch [17/100], Train Loss: 0.0097, Test Loss: 0.0053
Epoch [18/100], Train Loss: 0.0089, Test Loss: 0.0032
  -> Saved best ResNet model!
Epoch [19/100], Train Loss: 0.0091, Test Loss: 0.0047
Epoch [20/100], Train Loss: 0.0085, Test Loss: 0.0035
Epoch [21/100], Train Loss: 0.0081, Test Loss: 0.0048
Epoch [22/100], Train Loss: 0.0081, Test Loss: 0.0022
  -> Saved best ResNet model!
Epoch [23/100], Train Loss: 0.0087, Test Loss: 0.0036
Epoch [24/100], Train Loss: 0.0081, Test Loss: 0.0029
Epoch [25/100], Train Loss: 0.0081, Test Loss: 0.0020
  -> Saved best ResNet model!
Epoch [26/100], Train Loss: 0.0076, Test Loss: 0.0028
Epoch [27/100], Train Loss: 0.0077, Test Loss: 0.0024
Epoch [28/100], Train Loss: 0.0078, Test Loss: 0.0068
Epoch [29/100], Train Loss: 0.0075, Test Loss: 0.0025
Epoch [30/100], Train Loss: 0.0083, Test Loss: 0.0019
  -> Saved best ResNet model!
Epoch [31/100], Train Loss: 0.0080, Test Loss: 0.0026
Epoch [32/100], Train Loss: 0.0077, Test Loss: 0.0022
Epoch [33/100], Train Loss: 0.0071, Test Loss: 0.0035
Epoch [34/100], Train Loss: 0.0069, Test Loss: 0.0026
Epoch [35/100], Train Loss: 0.0066, Test Loss: 0.0019
Epoch [36/100], Train Loss: 0.0068, Test Loss: 0.0017
  -> Saved best ResNet model!
Epoch [37/100], Train Loss: 0.0072, Test Loss: 0.0024
Epoch [38/100], Train Loss: 0.0067, Test Loss: 0.0024
Epoch [39/100], Train Loss: 0.0065, Test Loss: 0.0027
Epoch [40/100], Train Loss: 0.0065, Test Loss: 0.0040
Epoch [41/100], Train Loss: 0.0071, Test Loss: 0.0026
Epoch [42/100], Train Loss: 0.0069, Test Loss: 0.0020
Epoch [43/100], Train Loss: 0.0063, Test Loss: 0.0022
Epoch [44/100], Train Loss: 0.0058, Test Loss: 0.0020
Epoch [45/100], Train Loss: 0.0062, Test Loss: 0.0045
Epoch [46/100], Train Loss: 0.0064, Test Loss: 0.0040
Epoch [47/100], Train Loss: 0.0061, Test Loss: 0.0030
Epoch [48/100], Train Loss: 0.0061, Test Loss: 0.0029
Epoch [49/100], Train Loss: 0.0064, Test Loss: 0.0029
Epoch [50/100], Train Loss: 0.0059, Test Loss: 0.0025
Epoch [51/100], Train Loss: 0.0055, Test Loss: 0.0014
  -> Saved best ResNet model!
Epoch [52/100], Train Loss: 0.0058, Test Loss: 0.0019
Epoch [53/100], Train Loss: 0.0056, Test Loss: 0.0024
Epoch [54/100], Train Loss: 0.0057, Test Loss: 0.0024
Epoch [55/100], Train Loss: 0.0059, Test Loss: 0.0030
Epoch [56/100], Train Loss: 0.0059, Test Loss: 0.0027
Epoch [57/100], Train Loss: 0.0053, Test Loss: 0.0016
Epoch [58/100], Train Loss: 0.0057, Test Loss: 0.0012
  -> Saved best ResNet model!
Epoch [59/100], Train Loss: 0.0054, Test Loss: 0.0021
Epoch [60/100], Train Loss: 0.0052, Test Loss: 0.0014
Epoch [61/100], Train Loss: 0.0050, Test Loss: 0.0019
Epoch [62/100], Train Loss: 0.0053, Test Loss: 0.0024
Epoch [63/100], Train Loss: 0.0056, Test Loss: 0.0018
Epoch [64/100], Train Loss: 0.0055, Test Loss: 0.0022
Epoch [65/100], Train Loss: 0.0051, Test Loss: 0.0022
Epoch [66/100], Train Loss: 0.0055, Test Loss: 0.0018
Epoch [67/100], Train Loss: 0.0052, Test Loss: 0.0024
Epoch [68/100], Train Loss: 0.0055, Test Loss: 0.0018
Epoch [69/100], Train Loss: 0.0049, Test Loss: 0.0021
Epoch [70/100], Train Loss: 0.0053, Test Loss: 0.0019
Epoch [71/100], Train Loss: 0.0049, Test Loss: 0.0019
Epoch [72/100], Train Loss: 0.0050, Test Loss: 0.0024
Epoch [73/100], Train Loss: 0.0058, Test Loss: 0.0022
Epoch [74/100], Train Loss: 0.0049, Test Loss: 0.0015
Epoch [75/100], Train Loss: 0.0051, Test Loss: 0.0019
Epoch [76/100], Train Loss: 0.0054, Test Loss: 0.0023
Epoch [77/100], Train Loss: 0.0047, Test Loss: 0.0016
Epoch [78/100], Train Loss: 0.0046, Test Loss: 0.0012
Epoch [79/100], Train Loss: 0.0046, Test Loss: 0.0012
Epoch [80/100], Train Loss: 0.0048, Test Loss: 0.0016
Epoch [81/100], Train Loss: 0.0049, Test Loss: 0.0015
Epoch [82/100], Train Loss: 0.0046, Test Loss: 0.0015
Epoch [83/100], Train Loss: 0.0045, Test Loss: 0.0013
Epoch [84/100], Train Loss: 0.0046, Test Loss: 0.0014
Epoch [85/100], Train Loss: 0.0045, Test Loss: 0.0019
Epoch [86/100], Train Loss: 0.0044, Test Loss: 0.0015
Epoch [87/100], Train Loss: 0.0047, Test Loss: 0.0017
Epoch [88/100], Train Loss: 0.0046, Test Loss: 0.0017
Epoch [89/100], Train Loss: 0.0043, Test Loss: 0.0011
  -> Saved best ResNet model!
Epoch [90/100], Train Loss: 0.0041, Test Loss: 0.0010
  -> Saved best ResNet model!
Epoch [91/100], Train Loss: 0.0040, Test Loss: 0.0013
Epoch [92/100], Train Loss: 0.0041, Test Loss: 0.0020
Epoch [93/100], Train Loss: 0.0045, Test Loss: 0.0013
Epoch [94/100], Train Loss: 0.0044, Test Loss: 0.0013
Epoch [95/100], Train Loss: 0.0039, Test Loss: 0.0015
Epoch [96/100], Train Loss: 0.0041, Test Loss: 0.0019
Epoch [97/100], Train Loss: 0.0043, Test Loss: 0.0015
Epoch [98/100], Train Loss: 0.0043, Test Loss: 0.0012
Epoch [99/100], Train Loss: 0.0042, Test Loss: 0.0015
Epoch [100/100], Train Loss: 0.0042, Test Loss: 0.0018
def evaluate_resnet():
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    # MUST MATCH TRAINING TRANSFORMS
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    test_dataset = ClockDataset("../data", subset='test', transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

    model = DigitalClockResNet().to(device)
    
    try:
        # Load the ResNet checkpoint
        model.load_state_dict(torch.load("checkpoints/digital_resnet_best.pth", map_location=device))
        print("ResNet Model loaded successfully!")
    except FileNotFoundError:
        print("Error: ResNet Checkpoint not found.")
        return

    model.eval()

    print("\n--- Visual Evaluation (ResNet) ---")
    print(f"{'Actual':<10} | {'Predicted':<10} | {'Diff'}")
    print("-" * 35)

    def denormalize_time(tensor_time):
        # Clamp ensures values don't go below 0 or above 1 due to minor noise
        tensor_time = torch.clamp(tensor_time, 0, 1) 
        h = int(tensor_time[0].item() * 23)
        m = int(tensor_time[1].item() * 59)
        s = int(tensor_time[2].item() * 59)
        return f"{h:02d}:{m:02d}:{s:02d}"

    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            if i >= 10: break
            
            img = batch['digital_img'].to(device)
            target = batch['time_label'].to(device)
            
            output = model(img)
            
            actual_str = denormalize_time(target[0])
            pred_str = denormalize_time(output[0])
            
            loss = torch.nn.functional.mse_loss(output, target).item()
            
            print(f"{actual_str:<10} | {pred_str:<10} | {loss:.6f}")

if __name__ == "__main__":
    evaluate_resnet()
ResNet Model loaded successfully!

--- Visual Evaluation (ResNet) ---
Actual     | Predicted  | Diff
-----------------------------------
06:05:26   | 06:07:26   | 0.000681
19:04:47   | 20:03:48   | 0.000172
11:38:40   | 11:36:41   | 0.000373
05:59:45   | 05:59:45   | 0.000187
18:01:10   | 16:02:08   | 0.003483
22:33:10   | 22:33:09   | 0.000146
23:52:57   | 21:56:55   | 0.003199
08:10:06   | 08:06:04   | 0.001694
13:38:16   | 13:37:15   | 0.000165
11:05:53   | 11:06:56   | 0.000714
# ===================== MODEL =====================

class DigitalClockClassifier(nn.Module):
    """
    ResNet18 backbone with 3 independent classification heads:
      - hour_head:   24 classes  (0–23)
      - minute_head: 60 classes  (0–59)
      - second_head: 60 classes  (0–59)

    Why classification instead of regression?
    Regression treats 12:59 and 13:00 as "close" in loss space,
    even though they are very different clock positions.
    Classification treats each digit combination as an independent class,
    so the model can learn crisp decision boundaries.
    """
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        feat = base.fc.in_features          # 512 for ResNet18
        base.fc = nn.Identity()             # Remove the original head
        self.backbone = base

        # Shared bottleneck (helps all 3 heads)
        self.bottleneck = nn.Sequential(
            nn.Linear(feat, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.hour_head   = nn.Linear(256, 24)
        self.minute_head = nn.Linear(256, 60)
        self.second_head = nn.Linear(256, 60)

    def forward(self, x):
        f = self.backbone(x)
        f = self.bottleneck(f)
        return self.hour_head(f), self.minute_head(f), self.second_head(f)

    def predict_time(self, x):
        """Returns integer (h, m, s) — use this at inference time."""
        h_logits, m_logits, s_logits = self.forward(x)
        h = h_logits.argmax(dim=1)
        m = m_logits.argmax(dim=1)
        s = s_logits.argmax(dim=1)
        return h, m, s
def train(data_dir='../data', num_epochs=80, batch_size=32, lr=1e-4):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        # More aggressive augmentation — helps distinguish similar digits
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # slight shift
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    transform_val = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_ds = ClockDataset(data_dir, subset='train', transform=transform)
    test_ds  = ClockDataset(data_dir, subset='test',  transform=transform_val)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0)

    model     = DigitalClockClassifier().to(device)
    # Label smoothing helps when similar digits are being confused (e.g. 52 vs 54)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Warmup for 5 epochs then cosine decay — prevents early bad local minima
    def lr_lambda(epoch):
        if epoch < 5:
            return (epoch + 1) / 5   # linear warmup
        # cosine decay from epoch 5 to num_epochs
        progress = (epoch - 5) / (num_epochs - 5)
        return 0.5 * (1 + math.cos(math.pi * progress))
    
    import math
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    os.makedirs('checkpoints', exist_ok=True)
    best_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            imgs   = batch['digital_img'].to(device)
            labels = batch['original_time'].to(device)
            h_true, m_true, s_true = labels[:,0], labels[:,1], labels[:,2]

            h_pred, m_pred, s_pred = model(imgs)
            # Weight minute and second loss higher — they are harder (60 classes vs 24)
            loss = (criterion(h_pred, h_true) +
                    2.0 * criterion(m_pred, m_true) +
                    2.0 * criterion(s_pred, s_true))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        model.eval()
        correct_h = correct_m = correct_s = correct_all = total = 0
        with torch.no_grad():
            for batch in test_loader:
                imgs   = batch['digital_img'].to(device)
                labels = batch['original_time'].to(device)
                h_true, m_true, s_true = labels[:,0], labels[:,1], labels[:,2]
                h_p, m_p, s_p = model.predict_time(imgs)
                correct_h   += (h_p == h_true).sum().item()
                correct_m   += (m_p == m_true).sum().item()
                correct_s   += (s_p == s_true).sum().item()
                correct_all += ((h_p == h_true) & (m_p == m_true) & (s_p == s_true)).sum().item()
                total += imgs.size(0)

        acc_all = 100 * correct_all / total
        print(f"Epoch [{epoch+1:>3}/{num_epochs}]  "
              f"Loss: {total_loss/len(train_loader):.4f}  lr: {current_lr:.5f}  |  "
              f"H: {100*correct_h/total:.0f}%  "
              f"M: {100*correct_m/total:.0f}%  "
              f"S: {100*correct_s/total:.0f}%  |  "
              f"ALL: {acc_all:.1f}%")

        if acc_all > best_acc:
            best_acc = acc_all
            torch.save(model.state_dict(), 'checkpoints/digital_reader_best.pth')
            print(f"  -> Saved! Best: {best_acc:.1f}%")
# ===================== EVALUATION =====================

def evaluate(data_dir='./clock_dataset'):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    test_ds = ClockDataset(data_dir, subset='test', transform=transform)
    loader  = DataLoader(test_ds, batch_size=1, shuffle=True, num_workers=0)

    model = DigitalClockClassifier().to(device)
    model.load_state_dict(torch.load('checkpoints/digital_reader_best.pth', map_location=device))
    model.eval()

    print(f"\n{'Actual':<12} {'Predicted':<12} {'Correct?'}")
    print("-" * 40)

    errors = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= 20: break
            imgs = batch['digital_img'].to(device)
            h_true, m_true, s_true = (batch['original_time'][:, j].item() for j in range(3))

            h_pred, m_pred, s_pred = model.predict_time(imgs)
            h_p, m_p, s_p = h_pred.item(), m_pred.item(), s_pred.item()

            actual = f"{h_true:02d}:{m_true:02d}:{s_true:02d}"
            pred   = f"{h_p:02d}:{m_p:02d}:{s_p:02d}"
            ok     = "OK" if actual == pred else "WRONG"
            print(f"{actual:<12} {pred:<12} {ok}")

            # Show first failure visually
            if ok == "WRONG" and len(errors) == 0:
                errors.append((imgs.cpu(), actual, pred))

    if errors:
        img_t, actual, pred = errors[0]
        plt.figure(figsize=(4, 4))
        plt.imshow(img_t[0].permute(1, 2, 0).clamp(0, 1))
        plt.title(f"Failure: actual={actual}  pred={pred}", color='red')
        plt.axis('off')
        plt.tight_layout()
        print("\nSaved failure example to reader_failure.png")
train(data_dir='./clock_dataset', num_epochs=80)
Epoch [  1/80]  Loss: 19.6795  lr: 0.00004  |  H: 5%  M: 1%  S: 4%  |  ALL: 0.0%
Epoch [  2/80]  Loss: 19.4567  lr: 0.00006  |  H: 6%  M: 2%  S: 1%  |  ALL: 0.0%
Epoch [  3/80]  Loss: 19.0985  lr: 0.00008  |  H: 14%  M: 4%  S: 3%  |  ALL: 0.0%
Epoch [  4/80]  Loss: 18.4730  lr: 0.00010  |  H: 9%  M: 7%  S: 6%  |  ALL: 0.0%
Epoch [  5/80]  Loss: 17.3353  lr: 0.00010  |  H: 10%  M: 12%  S: 10%  |  ALL: 0.0%
Epoch [  6/80]  Loss: 15.6319  lr: 0.00010  |  H: 17%  M: 20%  S: 17%  |  ALL: 0.5%
  -> Saved! Best: 0.5%
Epoch [  7/80]  Loss: 13.8673  lr: 0.00010  |  H: 22%  M: 27%  S: 26%  |  ALL: 1.0%
  -> Saved! Best: 1.0%
Epoch [  8/80]  Loss: 12.1015  lr: 0.00010  |  H: 31%  M: 31%  S: 36%  |  ALL: 4.0%
  -> Saved! Best: 4.0%
Epoch [  9/80]  Loss: 10.6282  lr: 0.00010  |  H: 40%  M: 38%  S: 44%  |  ALL: 7.5%
  -> Saved! Best: 7.5%
Epoch [ 10/80]  Loss: 9.1353  lr: 0.00010  |  H: 42%  M: 43%  S: 56%  |  ALL: 8.5%
  -> Saved! Best: 8.5%
Epoch [ 11/80]  Loss: 7.8314  lr: 0.00010  |  H: 54%  M: 48%  S: 60%  |  ALL: 13.5%
  -> Saved! Best: 13.5%
Epoch [ 12/80]  Loss: 6.8732  lr: 0.00010  |  H: 58%  M: 52%  S: 67%  |  ALL: 18.0%
  -> Saved! Best: 18.0%
Epoch [ 13/80]  Loss: 6.0456  lr: 0.00010  |  H: 64%  M: 52%  S: 74%  |  ALL: 21.5%
  -> Saved! Best: 21.5%
Epoch [ 14/80]  Loss: 5.3798  lr: 0.00010  |  H: 68%  M: 58%  S: 79%  |  ALL: 29.5%
  -> Saved! Best: 29.5%
Epoch [ 15/80]  Loss: 4.8495  lr: 0.00010  |  H: 70%  M: 55%  S: 82%  |  ALL: 27.0%
Epoch [ 16/80]  Loss: 4.4774  lr: 0.00009  |  H: 70%  M: 62%  S: 82%  |  ALL: 34.5%
  -> Saved! Best: 34.5%
Epoch [ 17/80]  Loss: 4.0326  lr: 0.00009  |  H: 84%  M: 60%  S: 86%  |  ALL: 39.5%
  -> Saved! Best: 39.5%
Epoch [ 18/80]  Loss: 3.7721  lr: 0.00009  |  H: 80%  M: 62%  S: 88%  |  ALL: 38.5%
Epoch [ 19/80]  Loss: 3.5723  lr: 0.00009  |  H: 82%  M: 63%  S: 88%  |  ALL: 44.0%
  -> Saved! Best: 44.0%
Epoch [ 20/80]  Loss: 3.4341  lr: 0.00009  |  H: 83%  M: 65%  S: 90%  |  ALL: 43.5%
Epoch [ 21/80]  Loss: 3.2844  lr: 0.00009  |  H: 87%  M: 66%  S: 87%  |  ALL: 47.5%
  -> Saved! Best: 47.5%
Epoch [ 22/80]  Loss: 3.1084  lr: 0.00009  |  H: 90%  M: 70%  S: 92%  |  ALL: 55.5%
  -> Saved! Best: 55.5%
Epoch [ 23/80]  Loss: 3.0131  lr: 0.00009  |  H: 86%  M: 67%  S: 91%  |  ALL: 50.5%
Epoch [ 24/80]  Loss: 2.9467  lr: 0.00008  |  H: 90%  M: 69%  S: 91%  |  ALL: 54.0%
Epoch [ 25/80]  Loss: 2.8430  lr: 0.00008  |  H: 90%  M: 68%  S: 92%  |  ALL: 54.5%
Epoch [ 26/80]  Loss: 2.8130  lr: 0.00008  |  H: 91%  M: 72%  S: 92%  |  ALL: 56.0%
  -> Saved! Best: 56.0%
Epoch [ 27/80]  Loss: 2.7808  lr: 0.00008  |  H: 92%  M: 72%  S: 92%  |  ALL: 58.0%
  -> Saved! Best: 58.0%
Epoch [ 28/80]  Loss: 2.7476  lr: 0.00008  |  H: 92%  M: 71%  S: 92%  |  ALL: 57.0%
Epoch [ 29/80]  Loss: 2.6677  lr: 0.00008  |  H: 93%  M: 74%  S: 92%  |  ALL: 60.5%
  -> Saved! Best: 60.5%
Epoch [ 30/80]  Loss: 2.6386  lr: 0.00008  |  H: 92%  M: 75%  S: 95%  |  ALL: 62.0%
  -> Saved! Best: 62.0%
Epoch [ 31/80]  Loss: 2.6287  lr: 0.00007  |  H: 94%  M: 74%  S: 93%  |  ALL: 61.5%
Epoch [ 32/80]  Loss: 2.5991  lr: 0.00007  |  H: 95%  M: 75%  S: 92%  |  ALL: 64.5%
  -> Saved! Best: 64.5%
Epoch [ 33/80]  Loss: 2.5768  lr: 0.00007  |  H: 92%  M: 78%  S: 94%  |  ALL: 64.5%
Epoch [ 34/80]  Loss: 2.5431  lr: 0.00007  |  H: 94%  M: 76%  S: 91%  |  ALL: 62.5%
Epoch [ 35/80]  Loss: 2.5579  lr: 0.00007  |  H: 95%  M: 76%  S: 92%  |  ALL: 64.5%
Epoch [ 36/80]  Loss: 2.5647  lr: 0.00006  |  H: 96%  M: 74%  S: 94%  |  ALL: 64.5%
Epoch [ 37/80]  Loss: 2.5437  lr: 0.00006  |  H: 96%  M: 78%  S: 94%  |  ALL: 69.0%
  -> Saved! Best: 69.0%
Epoch [ 38/80]  Loss: 2.5016  lr: 0.00006  |  H: 96%  M: 79%  S: 94%  |  ALL: 69.5%
  -> Saved! Best: 69.5%
Epoch [ 39/80]  Loss: 2.5049  lr: 0.00006  |  H: 95%  M: 77%  S: 94%  |  ALL: 67.0%
Epoch [ 40/80]  Loss: 2.4626  lr: 0.00006  |  H: 96%  M: 75%  S: 95%  |  ALL: 67.5%
Epoch [ 41/80]  Loss: 2.4747  lr: 0.00005  |  H: 96%  M: 76%  S: 94%  |  ALL: 66.0%
Epoch [ 42/80]  Loss: 2.4537  lr: 0.00005  |  H: 96%  M: 76%  S: 96%  |  ALL: 67.0%
Epoch [ 43/80]  Loss: 2.4489  lr: 0.00005  |  H: 96%  M: 78%  S: 95%  |  ALL: 69.0%
Epoch [ 44/80]  Loss: 2.4548  lr: 0.00005  |  H: 96%  M: 78%  S: 94%  |  ALL: 69.0%
Epoch [ 45/80]  Loss: 2.4501  lr: 0.00004  |  H: 95%  M: 78%  S: 94%  |  ALL: 67.0%
Epoch [ 46/80]  Loss: 2.4333  lr: 0.00004  |  H: 94%  M: 78%  S: 96%  |  ALL: 68.0%
Epoch [ 47/80]  Loss: 2.4635  lr: 0.00004  |  H: 95%  M: 78%  S: 94%  |  ALL: 68.0%
Epoch [ 48/80]  Loss: 2.4240  lr: 0.00004  |  H: 94%  M: 78%  S: 94%  |  ALL: 67.0%
Epoch [ 49/80]  Loss: 2.4428  lr: 0.00004  |  H: 95%  M: 80%  S: 93%  |  ALL: 69.0%
Epoch [ 50/80]  Loss: 2.4053  lr: 0.00003  |  H: 94%  M: 80%  S: 94%  |  ALL: 69.0%
Epoch [ 51/80]  Loss: 2.4054  lr: 0.00003  |  H: 96%  M: 81%  S: 95%  |  ALL: 72.5%
  -> Saved! Best: 72.5%
Epoch [ 52/80]  Loss: 2.4029  lr: 0.00003  |  H: 97%  M: 82%  S: 94%  |  ALL: 72.5%
Epoch [ 53/80]  Loss: 2.4075  lr: 0.00003  |  H: 96%  M: 83%  S: 96%  |  ALL: 74.5%
  -> Saved! Best: 74.5%
Epoch [ 54/80]  Loss: 2.4031  lr: 0.00003  |  H: 96%  M: 81%  S: 96%  |  ALL: 74.0%
Epoch [ 55/80]  Loss: 2.3804  lr: 0.00003  |  H: 96%  M: 80%  S: 96%  |  ALL: 72.0%
Epoch [ 56/80]  Loss: 2.3936  lr: 0.00002  |  H: 97%  M: 81%  S: 95%  |  ALL: 73.0%
Epoch [ 57/80]  Loss: 2.3912  lr: 0.00002  |  H: 98%  M: 80%  S: 95%  |  ALL: 73.0%
Epoch [ 58/80]  Loss: 2.3896  lr: 0.00002  |  H: 98%  M: 82%  S: 95%  |  ALL: 75.0%
  -> Saved! Best: 75.0%
Epoch [ 59/80]  Loss: 2.3713  lr: 0.00002  |  H: 98%  M: 84%  S: 94%  |  ALL: 75.5%
  -> Saved! Best: 75.5%
Epoch [ 60/80]  Loss: 2.3425  lr: 0.00002  |  H: 98%  M: 82%  S: 96%  |  ALL: 75.5%
Epoch [ 61/80]  Loss: 2.3708  lr: 0.00002  |  H: 97%  M: 82%  S: 96%  |  ALL: 75.0%
Epoch [ 62/80]  Loss: 2.3820  lr: 0.00001  |  H: 98%  M: 82%  S: 94%  |  ALL: 74.0%
Epoch [ 63/80]  Loss: 2.3583  lr: 0.00001  |  H: 98%  M: 82%  S: 94%  |  ALL: 74.5%
Epoch [ 64/80]  Loss: 2.3697  lr: 0.00001  |  H: 96%  M: 85%  S: 94%  |  ALL: 76.0%
  -> Saved! Best: 76.0%
Epoch [ 65/80]  Loss: 2.3789  lr: 0.00001  |  H: 96%  M: 84%  S: 96%  |  ALL: 75.0%
Epoch [ 66/80]  Loss: 2.3685  lr: 0.00001  |  H: 97%  M: 84%  S: 96%  |  ALL: 76.5%
  -> Saved! Best: 76.5%
Epoch [ 67/80]  Loss: 2.3682  lr: 0.00001  |  H: 97%  M: 82%  S: 96%  |  ALL: 74.5%
Epoch [ 68/80]  Loss: 2.3652  lr: 0.00001  |  H: 97%  M: 81%  S: 96%  |  ALL: 74.0%
Epoch [ 69/80]  Loss: 2.3637  lr: 0.00001  |  H: 97%  M: 81%  S: 95%  |  ALL: 73.0%
Epoch [ 70/80]  Loss: 2.3739  lr: 0.00000  |  H: 97%  M: 82%  S: 96%  |  ALL: 76.0%
Epoch [ 71/80]  Loss: 2.3548  lr: 0.00000  |  H: 96%  M: 82%  S: 95%  |  ALL: 74.0%
Epoch [ 72/80]  Loss: 2.3500  lr: 0.00000  |  H: 96%  M: 81%  S: 96%  |  ALL: 73.5%
Epoch [ 73/80]  Loss: 2.3528  lr: 0.00000  |  H: 96%  M: 82%  S: 96%  |  ALL: 74.5%
Epoch [ 74/80]  Loss: 2.3539  lr: 0.00000  |  H: 97%  M: 82%  S: 96%  |  ALL: 74.5%
Epoch [ 75/80]  Loss: 2.3540  lr: 0.00000  |  H: 97%  M: 83%  S: 96%  |  ALL: 76.0%
Epoch [ 76/80]  Loss: 2.3297  lr: 0.00000  |  H: 97%  M: 84%  S: 96%  |  ALL: 76.5%
Epoch [ 77/80]  Loss: 2.3705  lr: 0.00000  |  H: 96%  M: 84%  S: 96%  |  ALL: 76.5%
Epoch [ 78/80]  Loss: 2.3515  lr: 0.00000  |  H: 96%  M: 82%  S: 96%  |  ALL: 74.0%
Epoch [ 79/80]  Loss: 2.3229  lr: 0.00000  |  H: 97%  M: 82%  S: 96%  |  ALL: 75.5%
Epoch [ 80/80]  Loss: 2.3642  lr: 0.00000  |  H: 97%  M: 84%  S: 96%  |  ALL: 77.0%
  -> Saved! Best: 77.0%
---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[27], line 2
      1 train(data_dir='./clock_dataset', num_epochs=80)
----> 2 evaluate()

Cell In[22], line 10, in evaluate(data_dir)
      3 def evaluate(data_dir='../data'):
      4     transform = transforms.Compose([
      5         transforms.Resize((224, 224)),
      6         transforms.ToTensor(),
      7         transforms.Normalize(mean=[0.485, 0.456, 0.406],
      8                              std=[0.229, 0.224, 0.225]),
      9     ])
---> 10     test_ds = ClockDataset(data_dir, subset='test', transform=transform)
     11     loader  = DataLoader(test_ds, batch_size=1, shuffle=True, num_workers=0)
     13     model = DigitalClockClassifier().to(device)

File ~/Desktop/College/שנה ג/מבוא לDL/Project/src/dataset.py:13, in ClockDataset.__init__(self, root_dir, subset, transform)
     10 self.labels_path = os.path.join(self.root_dir, "labels.csv")
     12 if not os.path.exists(self.labels_path):
---> 13     raise FileNotFoundError(f"labels.csv not found at: {self.labels_path}")
     15 self.df = pd.read_csv(self.labels_path)
     16 self.transform = transform

FileNotFoundError: labels.csv not found at: ../data/test/labels.csv
evaluate()
Actual       Predicted    Correct?
----------------------------------------
19:23:51     19:23:51     OK
16:46:36     16:46:36     OK
02:57:26     02:57:26     OK
05:51:46     05:51:46     OK
21:28:23     21:23:23     WRONG
20:30:47     20:39:47     WRONG
06:26:25     06:20:25     WRONG
04:49:29     04:49:29     OK
17:36:23     17:36:23     OK
05:51:33     05:51:33     OK
17:20:55     17:20:55     OK
19:51:39     19:51:39     OK
11:22:28     11:22:28     OK
22:44:57     22:44:57     OK
14:39:10     14:39:10     OK
05:39:44     05:39:44     OK
15:42:38     15:42:38     OK
22:41:08     22:34:08     WRONG
11:28:34     11:28:34     OK
11:08:29     11:08:29     OK

Saved failure example to reader_failure.png

 